# DOCdemo 自動化フロー — Google Colab 版

**最終更新**: 2026-05-12
**目的**: Brainverse 管理画面への企業登録 → コンテンツ生成 → 納品URL取得を **クラウド (Google Colab) 上で実行** し、チームメンバー全員が同じ環境で再現できるようにする。

---

## チーム向け使い方 (初回のみ)

1. このノートブックを **Colab で開く** (GitHub URL を Colab で開くか、Drive 上で右クリック → アプリで開く → Google Colaboratory)
2. **左サイドバーの 🔑 (Secrets) アイコン** で以下の Secret を **必ず先に登録** し、それぞれ `Notebook access` を ON にする:
   - `DOCDEMO_LOGIN_EMAIL` — Brainverse のログインメール
   - `DOCDEMO_LOGIN_PASSWORD` — 同パスワード
   - `GITHUB_TOKEN` — GitHub の Personal Access Token (`repo` スコープ。private リポジトリ clone 用)
3. メニュー **ランタイム → すべて実行** (`Ctrl+F9`)
4. CSV / 納品URL / ログは Google Drive の `MyDrive/DOCdemo_Colab/` に保存される

> **チーム共有のコツ**: Drive 上の `MyDrive/DOCdemo_Colab` フォルダを、運用するメンバー全員に共有 (編集権限) しておくと、CSV を同じ実体で参照できる。

---

## 何が実行されるか (ローカル版 README と同じ動作)

1. CSV に並べた企業名を 1 社ずつ Brainverse 管理画面に登録
2. 企業 HP を Yahoo! 検索で自動特定 (URL内の企業ID候補が複数あれば「同名企業該当」で人間判断に委ねる)
3. HP 内部リンク + 求人サイトURL (マイナビ等) を収集してコンテンツ生成 (FAQ + 企業情報) に渡す
4. 生成内容を 2 段階保存し、対象企業のデータか自動検証 (Step 4-pre / 4-post)
5. 完成したフロントエンド公開URL (= **納品URL**) を CSV に書き戻し
6. 「企業名 + 納品URL」の 2 列簡易 CSV をクライアント納品用として自動生成


## 1. Google Drive をマウント

チーム共有用フォルダ `MyDrive/DOCdemo_Colab/` を作成・マウントする。

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE = '/content/drive/MyDrive/DOCdemo_Colab'
os.makedirs(f'{DRIVE_BASE}/data', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/logs', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/screenshots', exist_ok=True)
print(f'✅ Drive ベース: {DRIVE_BASE}')

Mounted at /content/drive
✅ Drive ベース: /content/drive/MyDrive/DOCdemo_Colab


## 2. リポジトリの取得 (常に最新版を pull)

GitHub の `shofukui-neo/DOCdemo_update_project` をクローン (2 回目以降は `git pull`)。
private リポジトリの場合は Colab Secret `GITHUB_TOKEN` を使用。

In [2]:
import os, subprocess

REPO_DIR  = '/content/DOCdemo_update_project'
REPO_HOST = 'github.com'
REPO_PATH = 'shofukui-neo/DOCdemo_update_project.git'

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception as e:
    print(f'⚠️ GITHUB_TOKEN Secret 未取得 ({type(e).__name__}) — public リポジトリ想定で続行')

if token:
    repo_url = f'https://{token}@{REPO_HOST}/{REPO_PATH}'
    print('🔑 GITHUB_TOKEN を使用してアクセス')
else:
    repo_url = f'https://{REPO_HOST}/{REPO_PATH}'

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    print('既存リポジトリを更新 (git pull)...')
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', 'HEAD'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    print('リポジトリを clone...')
    subprocess.run(['git', 'clone', '--depth', '1', repo_url, REPO_DIR], check=True)

print('\n現在のコミット:')
subprocess.run(['git', '-C', REPO_DIR, 'log', '-1', '--oneline'])

⚠️ GITHUB_TOKEN Secret 未取得 (SecretNotFoundError) — public リポジトリ想定で続行
リポジトリを clone...

現在のコミット:


CompletedProcess(args=['git', '-C', '/content/DOCdemo_update_project', 'log', '-1', '--oneline'], returncode=0)

## 3. 依存パッケージのインストール (初回 ~3 分)

`requirements.txt` + Playwright + Chromium + システム依存ライブラリをセットアップ。

In [3]:
!pip install -q -r /content/DOCdemo_update_project/requirements.txt nest_asyncio
!playwright install chromium
!playwright install-deps chromium 2>/dev/null || true
print('✅ Playwright + 依存パッケージ準備完了')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 MB 13.8 MB/s eta 0:00:00
170.4 MiB [] 0% 0.0s170.4 MiB [] 0% 44.7s170.4 MiB [] 0% 18.4s170.4 MiB [] 0% 15.3s170.4 MiB [] 0% 8.1s170.4 MiB [] 1% 4.7s170.4 MiB [] 2% 3.7s170.4 MiB [] 3% 3.1s170.4 MiB [] 3% 3.2s170.4 MiB [] 4% 3.1s170.4 MiB [] 5% 2.7s170.4 MiB [] 6% 2.6s170.4 MiB [] 7% 2.6s170.4 MiB [] 8% 2.4s170.4 MiB [] 9% 2.4s170.4 MiB [] 10% 2.4s170.4 MiB [] 11% 2.3s170.4 MiB [] 12% 2.3s170.4 MiB [] 12% 2.4s170.4 MiB [] 12% 2.5s170.4 MiB [] 13% 2.4s170.4 MiB [] 14% 2.4s170.4 MiB [] 14% 2.5s170.4 MiB [] 14% 2.6s170.4 MiB [] 15% 2.6s170.4 MiB [] 15% 2.7s170.4 MiB [] 16% 2.7s170.4 MiB [] 17% 2.6s170.4 MiB [] 17% 2.5s170.4 MiB [] 18% 2.5s170.4 MiB [] 18% 2.6s170.4 MiB [] 19% 2.6s170.4 MiB [] 20% 2.6s170.4 MiB [] 20% 2.5s170.4 MiB [] 21% 2.4s170.4 MiB [] 22% 2.4s170.4 MiB [] 23% 2.3s170.4 MiB [] 24% 2.3s170.4 MiB [] 24% 2.2s170.4 MiB [] 25% 2.2s170.4 MiB [] 26% 2.1s170.4 MiB [] 27% 2.1s170.4 MiB [] 28% 2.0s170.4 MiB [] 29% 2.0s170.4 MiB

## 4. 認証情報・環境変数のセット

**Colab Secrets** (左サイドバーの 🔑 アイコン) に以下を登録しておくこと (各項目「ノートブックのアクセス」を必ず ON):

| Secret 名 | 値 |
|---|---|
| `DOCDEMO_LOGIN_EMAIL` | Brainverse のログインメール |
| `DOCDEMO_LOGIN_PASSWORD` | 同パスワード |

Secrets が未登録の場合は、その場で手入力するフォールバックに切り替わります (一時的な実行のみ・コミット禁止)。

In [4]:
import os, sys, getpass

def _get_secret(name: str, prompt: str, secret_input: bool = False) -> str:
    """Colab Secrets から取得、未登録/未認可なら getpass で手入力にフォールバック。"""
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            print(f'  🔑 {name}: Colab Secrets から取得')
            return value
    except Exception as e:
        # SecretNotFoundError / NotebookAccessError / その他
        print(f'  ⚠️ {name}: Colab Secrets 未登録 ({type(e).__name__}) → 手入力にフォールバック')
        print(f'     恒久対応は左サイドバー 🔑 で {name} を登録し「ノートブックのアクセス」を ON にしてください')
    # フォールバック
    if secret_input:
        return getpass.getpass(f'{prompt}: ')
    return input(f'{prompt}: ').strip()

os.environ['DOCDEMO_LOGIN_EMAIL']    = _get_secret('DOCDEMO_LOGIN_EMAIL',    'Brainverse メール')
os.environ['DOCDEMO_LOGIN_PASSWORD'] = _get_secret('DOCDEMO_LOGIN_PASSWORD', 'Brainverse パスワード', secret_input=True)
os.environ['DOCDEMO_HEADLESS']       = 'true'     # Colab はヘッドレス必須
os.environ['DOCDEMO_LOG_LEVEL']      = 'INFO'

REPO_DIR = '/content/DOCdemo_update_project'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

print()
print('✅ 認証情報セット完了')
print('   cwd     :', os.getcwd())
print('   email   :', os.environ['DOCDEMO_LOGIN_EMAIL'])
print('   headless:', os.environ['DOCDEMO_HEADLESS'])

  ⚠️ DOCDEMO_LOGIN_EMAIL: Colab Secrets 未登録 (SecretNotFoundError) → 手入力にフォールバック
     恒久対応は左サイドバー 🔑 で DOCDEMO_LOGIN_EMAIL を登録し「ノートブックのアクセス」を ON にしてください
Brainverse メール: neocareer.dev@admin
  ⚠️ DOCDEMO_LOGIN_PASSWORD: Colab Secrets 未登録 (SecretNotFoundError) → 手入力にフォールバック
     恒久対応は左サイドバー 🔑 で DOCDEMO_LOGIN_PASSWORD を登録し「ノートブックのアクセス」を ON にしてください
Brainverse パスワード: ··········

✅ 認証情報セット完了
   cwd     : /content/DOCdemo_update_project
   email   : neocareer.dev@admin
   headless: true


## 5. 企業リストCSVを準備

**ケースA: 新規作成** — 下のセルの `COMPANY_NAMES` に企業名を並べて実行 → Drive に初期CSVが生成される
**ケースB: 既存CSVを使う** — `MyDrive/DOCdemo_Colab/data/company_list.csv` に CSV が既にあれば、それを引き続き使う (途中再開可能)

In [6]:
from pathlib import Path
from spreadsheet_manager import SpreadsheetManager

CSV_PATH = Path(f'{DRIVE_BASE}/data/company_list.csv')

# === 新規作成する場合はここに企業名を並べる ===
COMPANY_NAMES = [
    # "株式会社サンプル",
    # "テスト株式会社",
]

if not CSV_PATH.exists():
    if not COMPANY_NAMES:
        raise RuntimeError(
            f'CSV がありません: {CSV_PATH}\n'
            f'COMPANY_NAMES に企業名を記入して再実行するか、'
            f'Drive 上の {CSV_PATH} に既存CSVを配置してください。'
        )
    SpreadsheetManager.create_initial_csv(COMPANY_NAMES, csv_path=CSV_PATH)
    print(f'✅ 初期CSV作成: {CSV_PATH} ({len(COMPANY_NAMES)}社)')
else:
    print(f'📂 既存のCSVを使用: {CSV_PATH}')

import pandas as pd
df = pd.read_csv(CSV_PATH, encoding='utf-8-sig')
print(f'\n登録企業数: {len(df)}社')
print('ステータス別:')
for s, c in df['ステータス'].value_counts().items():
    print(f'  {s}: {c}社')
df[['企業名', 'ホームページURL', 'ステータス', '納品URL']].head(20)

RuntimeError: CSV がありません: /content/drive/MyDrive/DOCdemo_Colab/data/company_list.csv
COMPANY_NAMES に企業名を記入して再実行するか、Drive 上の /content/drive/MyDrive/DOCdemo_Colab/data/company_list.csv に既存CSVを配置してください。

## 6. 自動化フロー実行

1社あたり目安 **3〜4分**。Colab のセッションタイムアウト (連続 12 時間 / アイドル 90 分) を意識し、100社超は分割実行推奨。

- **全件処理**: 下記そのまま実行
- **1社だけテスト**: `TEST_MODE = True` / `TARGET_COMPANY = '株式会社サンプル'` をセット

> 既に「完了」になっている企業は自動的にスキップされる (再実行で続きから処理)。

In [7]:
TEST_MODE       = False    # 1社だけテストするなら True
TARGET_COMPANY  = None     # TEST_MODE=True のとき対象企業名 (部分一致)

import nest_asyncio
nest_asyncio.apply()

from orchestrator import Orchestrator, setup_logging
setup_logging()

orchestrator = Orchestrator(
    csv_path=CSV_PATH,
    headless=True,
    test_mode=TEST_MODE,
    target_company=TARGET_COMPANY,
)
await orchestrator.run()

INFO:orchestrator:============================================================


2026-05-12 11:16:37,980 [INFO] orchestrator: ============================================================


INFO:orchestrator:DOCdemo 自動化フロー 開始


2026-05-12 11:16:37,990 [INFO] orchestrator: DOCdemo 自動化フロー 開始


INFO:orchestrator:開始時刻: 2026-05-12 11:16:37


2026-05-12 11:16:37,992 [INFO] orchestrator: 開始時刻: 2026-05-12 11:16:37


INFO:orchestrator:============================================================


2026-05-12 11:16:37,996 [INFO] orchestrator: ============================================================


FileNotFoundError: 企業リストCSVが見つかりません: /content/drive/MyDrive/DOCdemo_Colab/data/company_list.csv
先に create_initial_csv() で初期CSVを作成してください。

## 7. 実行結果サマリ & 納品URL一覧

In [ ]:
import pandas as pd
from IPython.display import HTML, display

df = pd.read_csv(CSV_PATH, encoding='utf-8-sig')

print('=== ステータス別件数 ===')
for s, c in df['ステータス'].value_counts().items():
    print(f'  {s}: {c}社')

print('\n=== 完了企業の納品URL ===')
completed = df[df['ステータス'] == '完了'][['企業名', '企業ID', '納品URL']].copy()

def _link(u):
    if pd.isna(u) or not u:
        return ''
    return f'<a href="{u}" target="_blank">{u}</a>'

completed['納品URL'] = completed['納品URL'].apply(_link)
display(HTML(
    '<style>table {font-size:14px; border-collapse:collapse;}'
    'th, td {padding:6px 10px; border:1px solid #ccc;}'
    'a {color:#1a73e8; text-decoration:none;}</style>'
    + completed.to_html(escape=False, index=False)
))

## 8. クライアント納品用シンプルCSV (企業名 + 納品URL の 2 列)

orchestrator 実行時に自動生成されているが、手動で再生成することも可能。

In [ ]:
import csv as _csv
from pathlib import Path

src = Path(CSV_PATH)
stem = src.stem
if stem.endswith('_company_list'):
    out_stem = stem[:-len('_company_list')] + '_delivery_urls'
else:
    out_stem = stem + '_delivery_urls'
delivery_csv = src.parent / f'{out_stem}.csv'

with open(src, encoding='utf-8-sig') as f_in, \
     open(delivery_csv, 'w', encoding='utf-8-sig', newline='') as f_out:
    reader = _csv.DictReader(f_in)
    writer = _csv.writer(f_out)
    writer.writerow(['企業名', '納品URL'])
    for row in reader:
        writer.writerow([row.get('企業名', ''), row.get('納品URL', '')])

print(f'✅ クライアント納品用CSV: {delivery_csv}')

## 9. 検証チェックリスト (verification_checklist.md) を生成

QA チームが手動検証に使うチェックリストを生成 → Drive に保存。

In [ ]:
import subprocess, shutil
from pathlib import Path

result = subprocess.run(
    ['python', 'generate_checklist.py'],
    cwd=REPO_DIR, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

src_md = Path(REPO_DIR) / 'verification_checklist.md'
dst_md = Path(DRIVE_BASE) / 'verification_checklist.md'
if src_md.exists():
    shutil.copy(src_md, dst_md)
    print(f'✅ チェックリストを Drive にコピー: {dst_md}')

## 10. ログ・スクリーンショットを Drive に保存

Colab ランタイムは終了時に消えるため、実行履歴・エラー解析用ファイルを Drive に書き戻す。

In [ ]:
import shutil, time
from datetime import datetime
from pathlib import Path

ts = datetime.now().strftime('%Y%m%d_%H%M%S')

src_log = Path(REPO_DIR) / 'logs' / 'automation.log'
dst_log = Path(DRIVE_BASE) / 'logs' / f'automation_{ts}.log'
if src_log.exists():
    shutil.copy(src_log, dst_log)
    print(f'✅ ログ保存: {dst_log}')

# 直近 24h に作成されたスクリーンショットだけ Drive にコピー (容量節約)
src_ss = Path(REPO_DIR) / 'screenshots'
dst_ss = Path(DRIVE_BASE) / 'screenshots'
dst_ss.mkdir(parents=True, exist_ok=True)
cutoff = time.time() - 86400
new_files = [p for p in src_ss.glob('*') if p.is_file() and p.stat().st_mtime > cutoff]
for p in new_files:
    shutil.copy(p, dst_ss / p.name)
print(f'✅ 直近24hのスクショ {len(new_files)} 件を Drive に保存: {dst_ss}')

---
## 11. 「同名企業該当」(URL企業ID不一致) になった企業への対応

検索結果に複数のドメインがあり、URL内の企業IDが一致しない候補が複数検出された場合、ステータスは `同名企業該当` になる。

**対応手順:**

1. Drive 上の `MyDrive/DOCdemo_Colab/data/company_list.csv` を **Google スプレッドシート** で開く (右クリック → アプリで開く → Google スプレッドシート)
2. ステータス列が `同名企業該当` の行を見つける
3. `URL候補` 列を確認 — パイプ `|` 区切りで最大 5 件のURLが入っている

   例:
   ```
   https://akiyama-group.com|https://kei-ichiman.com|https://www.c-c-akiyama.com
   ```

4. 各候補を実際にブラウザで開き、対象企業の公式サイトを判別
5. 正しい URL を `ホームページURL` 列に貼り付け
6. ファイル → ダウンロード → カンマ区切り形式 で保存し、Drive の同じ場所に `company_list.csv` として上書き
   - または Google スプレッドシートで保存 → 「6. 自動化フロー実行」セルを再実行
7. 該当企業のみ続きから処理される (他の `完了` 企業は自動スキップ)

> Google スプレッドシートで開いた CSV を直接編集する場合、ファイル形式が変わらないよう注意 (必ずカンマ区切り CSV で書き出し)。

---

## 12. トラブルシューティング

| 症状 | 原因 | 対処 |
|---|---|---|
| `SecretNotFoundError` | Colab Secrets 未登録 / Notebook access が OFF | 左サイドバー 🔑 で Secret を登録し、Notebook access をオン |
| `git clone` で 403 | private リポジトリで `GITHUB_TOKEN` が無い/期限切れ | GitHub で `repo` スコープの PAT を発行し Secret に登録 |
| `playwright install` が途中で失敗 | 一時的ネットワーク or Colab 制限 | セルを再実行 |
| ログイン失敗 | `DOCDEMO_LOGIN_PASSWORD` が古い | 管理担当者から最新値を取得し Secret 更新 |
| `same-name` が大量発生 | 検索ノイズが多い業種 | 「11. 同名企業該当への対応」の手順で手動補正 |
| ランタイムが切れた | 12h / アイドル90分のタイムアウト | 既処理分は CSV に残るので「6. 自動化フロー実行」を再実行 |
| Drive のCSVが更新されない | Drive の同期遅延 | 数十秒待つ or `drive.flush_and_unmount()` 後に再マウント |
| `Browser closed unexpectedly` | Playwright が落ちた | セル再実行 (CSV に途中ステータスが残っているので続きから) |

詳細ログは Drive の `logs/automation_<timestamp>.log` を確認。エラー時のスクリーンショットは `screenshots/` に保存される。

---

**お問い合わせ**: 自動化スクリプトのバグ → リポジトリ管理者 (Slack: #docdemo-auto) / Brainverse 認証 → 管理担当者